# 문제 4~5~6: Parquet 저장, Window Function, Broadcast Join

한 Colab 노트북 안에서 문제 4, 문제 5, 문제 6 코드를 별도 셀로 실행합니다. 문제 4에서 mock S3에 저장한 Parquet 객체를 문제 5와 문제 6에서 이어서 사용합니다.

In [ ]:
# Colab 실행 환경 준비
%pip install -q pyspark==3.5.3 boto3 moto==5.0.18

## 공통 준비 코드

In [ ]:
import logging
import os
import shutil
import sys
from pathlib import Path

import boto3
from moto import mock_aws
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import avg, broadcast, col, count, rank, round, sum, to_date, when
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType, TimestampType


logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s %(message)s",
    stream=sys.stdout,
    force=True,
)
logger = logging.getLogger("problem4_5_sensor_window")

# 제출 안내에 맞춰 실제 AWS 키 대신 마스킹 값을 사용합니다.
os.environ["AWS_ACCESS_KEY_ID"] = "**"
os.environ["AWS_SECRET_ACCESS_KEY"] = "**"
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

spark = (
    SparkSession.builder
    .appName("DE_5_week_problem4_5_sensor_window")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

bucket_name = "de-5-week-mock-bucket"
s3_prefix = "sensor/parquet/"
q4_s3_path = f"s3a://{bucket_name}/{s3_prefix}"
local_parquet_dir = Path("/content/sensor_parquet_output") if Path("/content").exists() else Path("sensor_parquet_output")
restored_parquet_dir = Path("/content/sensor_parquet_from_mock_s3") if Path("/content").exists() else Path("sensor_parquet_from_mock_s3")

sensor_schema = StructType([
    StructField("machine_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("vibration", DoubleType(), True),
    StructField("pressure", DoubleType(), True),
    StructField("status", StringType(), True),
])

equipment_schema = StructType([
    StructField("machine_id", StringType(), True),
    StructField("machine_name", StringType(), True),
    StructField("factory", StringType(), True),
    StructField("line_no", IntegerType(), True),
    StructField("install_date", StringType(), True),
])


def find_sensor_csv():
    candidates = [
        Path("sensor_logs.csv"),
        Path("/content/sensor_logs.csv"),
        Path("/Users/hanjeonghyun/Downloads/data/sensor_logs.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("sensor_logs.csv 파일을 Colab /content에 업로드하세요.")


def find_equipment_csv():
    candidates = [
        Path("equipment.csv"),
        Path("/content/equipment.csv"),
        Path("/Users/hanjeonghyun/Downloads/data/equipment.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("equipment.csv 파일을 Colab /content에 업로드하세요.")


def reset_dir(path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


if "MOTO_AWS_MOCK" not in globals():
    MOTO_AWS_MOCK = mock_aws()
    MOTO_AWS_MOCK.start()

s3 = boto3.client("s3", region_name="us-east-1")
existing_buckets = [bucket["Name"] for bucket in s3.list_buckets().get("Buckets", [])]
if bucket_name not in existing_buckets:
    s3.create_bucket(Bucket=bucket_name)

print("공통 준비 완료")

## 문제 4 코드: CSV 스키마 지정, Parquet 변환, mock S3 저장

In [ ]:
try:
    csv_path = find_sensor_csv()
    print(f"CSV 입력 파일: {csv_path}")

    df = spark.read.csv(str(csv_path), header=True, schema=sensor_schema)
    print("명시적 StructType 스키마")
    df.printSchema()
    df.show(5, truncate=False)

    df_with_date = df.withColumn("date", to_date(col("timestamp")))
    print("date 컬럼 추가 확인")
    df_with_date.show(3, truncate=False)

    print(f"Parquet 저장 대상: {q4_s3_path}")
    if local_parquet_dir.exists():
        shutil.rmtree(local_parquet_dir)
    df_with_date.write.mode("overwrite").partitionBy("machine_id", "date").parquet(str(local_parquet_dir))
    first_count = spark.read.parquet(str(local_parquet_dir)).count()

    old_objects = s3.list_objects_v2(Bucket=bucket_name, Prefix=s3_prefix).get("Contents", [])
    if old_objects:
        s3.delete_objects(Bucket=bucket_name, Delete={"Objects": [{"Key": obj["Key"]} for obj in old_objects]})

    for file_path in local_parquet_dir.rglob("*"):
        if file_path.is_file():
            key = s3_prefix + file_path.relative_to(local_parquet_dir).as_posix()
            s3.put_object(Bucket=bucket_name, Key=key, Body=file_path.read_bytes())

    listed = s3.list_objects_v2(Bucket=bucket_name, Prefix=s3_prefix).get("Contents", [])
    print("S3 저장 확인: Key / LastModified / Size")
    for obj in listed[:10]:
        print(f"Key: {obj['Key']}")
        print(f"LastModified: {obj['LastModified']}")
        print(f"Size: {obj['Size']} bytes")

    df_with_date.write.mode("overwrite").partitionBy("machine_id", "date").parquet(str(local_parquet_dir))
    second_count = spark.read.parquet(str(local_parquet_dir)).count()
    print("멱등성 검증")
    print(f"1회차 row count: {first_count}")
    print(f"2회차 row count: {second_count}")
    print(f"동일 여부: {first_count == second_count}")
    print("mode('overwrite')는 같은 경로의 기존 Parquet 결과를 지우고 다시 쓰므로 재실행 시 중복 누적을 방지합니다.")
    logger.info("문제 4 처리 완료")

except Exception as exc:
    logger.exception("문제 4 처리 실패: %s", exc)
    raise

## 문제 5 코드: 문제 4의 mock S3 Parquet 로딩 및 Window Function

In [ ]:
try:
    listed = s3.list_objects_v2(Bucket=bucket_name, Prefix=s3_prefix).get("Contents", [])
    if not listed:
        raise RuntimeError("문제 4 코드 셀을 먼저 실행해 mock S3에 Parquet를 저장하세요.")

    reset_dir(restored_parquet_dir)
    for obj in listed:
        key = obj["Key"]
        if key.endswith("/"):
            continue
        target = restored_parquet_dir / key.removeprefix(s3_prefix)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(s3.get_object(Bucket=bucket_name, Key=key)["Body"].read())

    print(f"mock S3 Parquet 로딩 경로: {q4_s3_path}")
    sensor_df = spark.read.parquet(str(restored_parquet_dir))
    print(f"로딩 row count: {sensor_df.count()}")

    timestamp_window = Window.partitionBy("machine_id").orderBy("timestamp")
    vibration_desc_window = Window.partitionBy("machine_id").orderBy(col("vibration").desc())
    temp_rows_window = timestamp_window.rowsBetween(-9, 0)
    cumulative_window = timestamp_window.rowsBetween(Window.unboundedPreceding, 0)

    feature_df = (
        sensor_df
        .withColumn("temp_moving_avg", avg("temperature").over(temp_rows_window))
        .withColumn("vibration_rank", rank().over(vibration_desc_window))
        .withColumn("error_cumsum", sum(when(col("status") == "오류", 1).otherwise(0)).over(cumulative_window))
    )

    anomaly_df = feature_df.where(
        (col("temp_moving_avg") > 85) | (col("vibration_rank") <= 5)
    ).select(
        "machine_id",
        "timestamp",
        "status",
        "temp_moving_avg",
        "vibration_rank",
        "error_cumsum",
    )

    print(f"이상 감지 건수: {anomaly_df.count()}")
    anomaly_df.show(20, truncate=False)
    logger.info("문제 5 처리 완료")

except Exception as exc:
    logger.exception("문제 5 처리 실패: %s", exc)
    raise

## 문제 6 코드: Broadcast Join 및 공장별 불량률 집계

In [ ]:
try:
    listed = s3.list_objects_v2(Bucket=bucket_name, Prefix=s3_prefix).get("Contents", [])
    if not listed:
        raise RuntimeError("문제 4 코드 셀을 먼저 실행해 mock S3에 Parquet를 저장하세요.")

    reset_dir(restored_parquet_dir)
    for obj in listed:
        key = obj["Key"]
        if key.endswith("/"):
            continue
        target = restored_parquet_dir / key.removeprefix(s3_prefix)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(s3.get_object(Bucket=bucket_name, Key=key)["Body"].read())

    equipment_path = find_equipment_csv()
    print(f"mock S3 Parquet 로딩 경로: {q4_s3_path}")
    print(f"equipment 입력 파일: {equipment_path}")

    df_logs = spark.read.parquet(str(restored_parquet_dir))
    df_equip = spark.read.csv(str(equipment_path), header=True, schema=equipment_schema)
    print("equipment schema + sample")
    df_equip.printSchema()
    df_equip.show(10, truncate=False)

    print(f"join 전 파티션 수: {df_logs.rdd.getNumPartitions()}")
    df_joined = df_logs.join(broadcast(df_equip), "machine_id")
    print(f"join 후 파티션 수: {df_joined.rdd.getNumPartitions()}")
    df_repartitioned = df_joined.repartition(4)
    print(f"repartition(4) 후 파티션 수: {df_repartitioned.rdd.getNumPartitions()}")

    factory_error_df = (
        df_repartitioned
        .groupBy("factory")
        .agg(
            count("*").alias("total_logs"),
            sum(when(col("status") == "오류", 1).otherwise(0)).alias("error_count"),
        )
        .withColumn("error_rate_pct", round(col("error_count") / col("total_logs") * 100, 2))
        .orderBy("factory")
    )

    print("공장별 불량률 집계")
    factory_error_df.show(truncate=False)

    result_prefix = "sensor/factory_error_rate/"
    result_s3_path = f"s3a://{bucket_name}/{result_prefix}"
    result_output_dir = Path("/content/factory_error_rate_output") if Path("/content").exists() else Path("factory_error_rate_output")
    if result_output_dir.exists():
        shutil.rmtree(result_output_dir)
    factory_error_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(str(result_output_dir))
    print(f"공장별 불량률 CSV 저장 대상: {result_s3_path}")

    old_result_objects = s3.list_objects_v2(Bucket=bucket_name, Prefix=result_prefix).get("Contents", [])
    if old_result_objects:
        s3.delete_objects(Bucket=bucket_name, Delete={"Objects": [{"Key": obj["Key"]} for obj in old_result_objects]})

    for file_path in result_output_dir.rglob("*"):
        if file_path.is_file():
            key = result_prefix + file_path.relative_to(result_output_dir).as_posix()
            s3.put_object(Bucket=bucket_name, Key=key, Body=file_path.read_bytes())

    result_objects = s3.list_objects_v2(Bucket=bucket_name, Prefix=result_prefix).get("Contents", [])
    print("S3 저장 확인: Key / LastModified / Size")
    for obj in result_objects:
        print(f"Key: {obj['Key']}")
        print(f"LastModified: {obj['LastModified']}")
        print(f"Size: {obj['Size']} bytes")

    print("broadcast를 적용한 이유: equipment는 설비 수가 적은 소규모 마스터 테이블이므로 각 executor에 복사해 shuffle 비용을 줄일 수 있습니다.")
    logger.info("문제 6 처리 완료")

except Exception as exc:
    logger.exception("문제 6 처리 실패: %s", exc)
    raise